In [52]:
import pandas as pd

In [53]:
caminho = "../data/apple_products_pricing_2020_2026.csv"
df_apple = pd.read_csv(caminho)
df_apple.shape

(80000, 14)

In [54]:
df_apple.columns

Index(['Date', 'Platform', 'Product_Category', 'Model_Name', 'Condition',
       'Launch_Price_USD', 'Launch_Price_INR', 'Current_Price_USD',
       'Current_Price_INR', 'Discount_Pct', 'Sale_Event', 'Stock_Status',
       'Rating', 'Reviews_Count'],
      dtype='str')

# Nível 1: agregar e agrupar (passo a passo)
## Exercício 1: agregar uma coluna
**Contexto:** antes de agrupar, o cálculo mais simples é resumir uma coluna inteira num número.

1. Na coluna `Current_Price_USD`: média (`.mean()`), mediana (`.median()`), maior (`.max()`) e menor (`.min()`).
2. Rode `.count()` na coluna `Sale_Event` e compare com `len(df_apple)`.

✅ No passo 2, os dois números são bem diferentes. Por quê? Anote sua hipótese antes do gabarito.

In [ ]:
data_price = df_apple["Current_Price_USD"].agg(["mean", "median", "max", "min"]).round(2)
print(f"A média é: $ {data_price["mean"]}")
print(f"A mediana é: $ {data_price["median"]}")
print(f"O maior valor é: $ {data_price["max"]}")
print(f"O menor valor é: $ {data_price["min"]}")

A média é: $ 782.77
A mediana é: $ 699.74
O maior valor é: $ 2038.97
O menor valor é: $ 109.93


mean       782.77
median     699.74
max       2038.97
min        109.93
Name: Current_Price_USD, dtype: float64

In [56]:
contagem = df_apple["Sale_Event"].count()
total_linhas  = len(df_apple)

subtracao = total_linhas  - contagem
print(f"O valor de .count() para a coluna Sale_Evento é: {contagem}")
print(f"O valor de len(df_apple) é: {total_linhas }")
print(f"len(df_apple) é {subtracao} maior porque .count() conta apenas as linhas que tem informação na coluna Sale_Event. Enquanto que len(df_apple) conta todas as linhas do DataFrame")

O valor de .count() para a coluna Sale_Evento é: 6649
O valor de len(df_apple) é: 80000
len(df_apple) é 73351 maior porque .count() conta apenas as linhas que tem informação na coluna Sale_Event. Enquanto que len(df_apple) conta todas as linhas do DataFrame


Eu acredito que os dois códigos apresentam valores diferentes porque o `df_apple["Sale_Event"].count()`, está contando apenas as linhas que têm evento. Enquanto que o `len(df_apple)` está contando todas as linhas do dataframe.

## Exercício 2: o loop antes do groupby

**Objetivo:** o preço atual médio de cada categoria (`Product_Category`), do jeito do exercício 7 do fundão.

1. `unique()` na coluna de categoria.
2. Um `for` que filtra cada categoria e calcula a média de `Current_Price_USD`.
3. Mostre com f-string: `iPhone: US$ 758.67`.

✅ Quatro linhas, uma por categoria.

In [57]:
lista_produto = df_apple["Product_Category"].unique()

for produto in lista_produto:
    filtro_produto = df_apple["Product_Category"] == produto
    df_produto = df_apple[filtro_produto]
    valor_produto = df_produto["Current_Price_USD"].mean()
    print(f"{produto}: US$ {valor_produto:,.2f}")

Watch: US$ 398.64
iPad: US$ 573.08
iPhone: US$ 758.67
Mac: US$ 1,382.49


## Exercício 3: o mesmo com groupby

1. Faça o exercício 2 numa linha: agrupe por `Product_Category`, pegue `Current_Price_USD`, tire a média.
2. Compare com os números do exercício 2.
3. Guarde o resultado numa variável e confira com `type()`.

✅ Mesmos números; o tipo é Series. Repare na ordem das categorias: `Mac, Watch, iPad, iPhone`. Não é a ordem do loop. Por quê?

In [58]:
series_products = df_apple.groupby("Product_Category", sort=False)["Current_Price_USD"].mean().round(2)
print(series_products)
print(type(series_products))

Product_Category
Watch      398.64
iPad       573.08
iPhone     758.67
Mac       1382.49
Name: Current_Price_USD, dtype: float64
<class 'pandas.Series'>


A ordem é diferente porque o groupby ordena as categorias por padrão (sort=True), com maiúsculas antes de minúsculas. O loop seguiu a ordem de aparição no CSV (unique()). Com sort=False, o groupby também usa a ordem de aparição.

## Exercício 4: abrir as pilhas

1. Rode o mesmo `groupby` **sem** o `.mean()` e veja o que aparece.
2. Pegue uma pilha: `df_apple.groupby("Product_Category").get_group("iPad")` e confira o `.shape`.
3. Volte ao exercício 3 e troque `.mean()` por `.max()`, depois `.min()`, depois `.count()`.

✅ No passo 1, o texto `<... SeriesGroupBy object at 0x...>`. No passo 2, o número de linhas é o mesmo que o `value_counts()` mostra para iPad no caderno.

In [59]:
# series_products = df_apple.groupby("Product_Category", sort=False)["Current_Price_USD"]

# df_apple.groupby("Product_Category").get_group("iPad").shape

print(df_apple.groupby("Product_Category", sort=False)["Current_Price_USD"].max().round(2))
print(df_apple.groupby("Product_Category", sort=False)["Current_Price_USD"].min().round(2))
print(df_apple.groupby("Product_Category", sort=False)["Current_Price_USD"].count().round(2))

Product_Category
Watch      814.94
iPad      1222.89
iPhone    1222.98
Mac       2038.97
Name: Current_Price_USD, dtype: float64
Product_Category
Watch     137.76
iPad      109.93
iPhone    226.61
Mac       332.76
Name: Current_Price_USD, dtype: float64
Product_Category
Watch     17865
iPad      15526
iPhone    28589
Mac       18020
Name: Current_Price_USD, dtype: int64


1. Rodei o `df_apple.groupby("Product_Category", sort=False)["Current_Price_USD"]`e apareceu isto: <pandas.api.typing.SeriesGroupBy object at 0x00000222AA203490>
2. Fiz rodei o 'df_apple.groupby("Product_Category").get_group("iPad").shape' e apareceu isto: (15526, 14). São as linhas e colunas do groupby apenas do grupo "iPad".
3. Feito no código acima.


## Exercício 5: count x size
Peça nova: .size() conta quantas linhas existem em cada grupo, sem olhar coluna nenhuma. .count() conta quantos valores preenchidos existem na coluna escolhida. A diferença entre os dois são os vazios.

df_apple.groupby("Platform")["Sale_Event"].count()

df_apple.groupby("Platform").size()

✅ Números muito diferentes. Ligue isso ao exercício 1.

In [60]:
df_apple.groupby("Platform")["Sale_Event"].count()

Platform
Amazon      3792
Flipkart    2857
Name: Sale_Event, dtype: int64

In [61]:
df_apple.groupby("Platform").size()

Platform
Amazon      39957
Flipkart    40043
dtype: int64

O `.count()` precisa de uma coluna: ele conta quantas linhas têm valor preenchido nela. Já o `.size()` não olha coluna nenhuma e conta todas as linhas do grupo, preenchidas ou vazias.

No exercício 1, `.count()` e `len()` foram aplicados ao DataFrame inteiro. Aqui o `groupby("Platform")` reparte a contagem entre Amazon e Flipkart. Somando os grupos, o `.count()` dá 6649 e o `.size()` dá 80000, os mesmos valores do exercício 1. Os dois métodos contam do mesmo jeito de antes, só que por pilha.

# Nível 2: as formas do resultado (objetivo + dica)
## Exercício 6: os 5 modelos mais caros
Objetivo: o preço atual médio de cada modelo (Model_Name), mostrando só os 5 mais caros.

Peça nova: numa Series, o sort_values não precisa do by, porque só há uma coluna de valores: .sort_values(ascending=False).

💡 groupby, .mean(), sort_values e head(5), encadeados.

✅ Quatro MacBook Pro e um iPhone.

In [62]:
df_apple.groupby("Model_Name")["Current_Price_USD"].mean().sort_values(ascending=False).head(5).round(2)

Model_Name
MacBook Pro 14-inch M4 Pro 512GB    1826.99
MacBook Pro 14-inch M3 Pro 512GB    1775.95
MacBook Pro 14-inch M2 Pro 512GB    1746.50
MacBook Pro 14-inch M1 Pro 512GB    1686.46
iPhone 15 Pro Max 256GB             1025.50
Name: Current_Price_USD, dtype: float64

## Exercício 7: duas chaves
Objetivo: o preço atual médio por plataforma e por categoria.

💡 A chave vira uma lista dentro dos parênteses. Depois aplique .reset_index() e compare type() e .shape antes e depois.

✅ 8 linhas (2 plataformas x 4 categorias). Antes do reset_index, Series; depois, DataFrame (8, 3). A plataforma faz diferença no preço?

In [63]:
lista = ['Platform', 'Product_Category']

df_apple.groupby(lista)["Current_Price_USD"].mean().round(2)

Platform  Product_Category
Amazon    Mac                 1386.41
          Watch                397.49
          iPad                 569.08
          iPhone               757.21
Flipkart  Mac                 1378.50
          Watch                399.77
          iPad                 577.11
          iPhone               760.13
Name: Current_Price_USD, dtype: float64

In [64]:
df_apple.groupby(lista)["Current_Price_USD"].mean().reset_index().round(2)

,Platform,Product_Category,Current_Price_USD
0,Amazon,Mac,1386.41
1,Amazon,Watch,397.49
2,Amazon,iPad,569.08
3,Amazon,iPhone,757.21
4,Flipkart,Mac,1378.50
5,Flipkart,Watch,399.77
6,Flipkart,iPad,577.11
7,Flipkart,iPhone,760.13


In [65]:
type(df_apple.groupby(lista)["Current_Price_USD"].mean().round(2))

pandas.Series

In [66]:
df_apple.groupby(lista)["Current_Price_USD"].mean().round(2).shape

(8,)

In [67]:
type(df_apple.groupby(lista)["Current_Price_USD"].mean().reset_index().round(2))

pandas.DataFrame

In [68]:
df_apple.groupby(lista)["Current_Price_USD"].mean().reset_index().round(2).shape

(8, 3)

**Reset index: antes e depois**

Sem o `.reset_index()`, o resultado é uma `Series` com `.shape` de `(8,)`: são 8 valores, e `Platform` e `Product_Category` ficam no índice, não como colunas. Com o `.reset_index()`, o resultado vira um `DataFrame` com `.shape` de `(8, 3)`: as mesmas 8 linhas, agora com 3 colunas (`Platform`, `Product_Category` e `Current_Price_USD`), porque o `.reset_index()` move as chaves do índice para dentro da tabela.

**A plataforma faz diferença no preço?**

Pela média por categoria, não. As diferenças entre Amazon e Flipkart são pequenas, de cerca de US$ 2 a US$ 8, ou seja, menos de 2% do preço de cada categoria. Também não há um padrão: a Amazon é mais cara só em Mac, e o Flipkart é mais caro em Watch, iPad e iPhone. Essa conclusão vale para a média por categoria, que mistura modelos, condições e datas. Um recorte mais fino poderia mostrar algo que a média esconde.

## Exercício 8: duas colunas de valor e uma conta entre elas
Objetivo: por categoria, a média de Launch_Price_USD e a de Current_Price_USD lado a lado. Depois, quanto cada categoria perdeu de valor, em dólares e em percentual.

💡 A seleção vira uma lista dentro dos colchetes. Guarde o resultado (é um DataFrame) e crie nele queda_usd (lançamento menos atual) e queda_pct (queda_usd dividida pelo lançamento, vezes 100). É o segundo caso da tabela "Contas entre Series".

✅ Compare a categoria que mais perde em dólares com a que mais perde em percentual.

In [122]:
lista = ["Launch_Price_USD", "Current_Price_USD"]
df_valores = df_apple.groupby("Product_Category")[lista].mean().round(2)
df_valores["queda_usd"] = df_valores["Launch_Price_USD"] - df_valores["Current_Price_USD"]
df_valores["queda_pct"] = ((df_valores["queda_usd"] / df_valores["Launch_Price_USD"]) * 100).round(1)

maior_queda_usd_categoria = df_valores.sort_values("queda_usd", ascending=False).head(1).index[0]
maior_queda_usd = df_valores.sort_values("queda_usd", ascending=False).loc[maior_queda_usd_categoria, "queda_usd"].round(2)

print(f"A categoria que mais perde em dólares é {maior_queda_usd_categoria} com uma perda de US$ {maior_queda_usd}")

maior_queda_pct_categoria = df_valores.sort_values("queda_pct", ascending=False).head(1).index[0]
maior_queda_pct = df_valores.sort_values("queda_pct", ascending=False).loc[maior_queda_pct_categoria, "queda_pct"].round(2)

print(f"A categoria que mais perde percentual é {maior_queda_pct_categoria} com uma queda de {maior_queda_pct}%")

A categoria que mais perde em dólares é Mac com uma perda de US$ 229.82
A categoria que mais perde percentual é Watch com uma queda de 25.3%


**Queda em dólares x queda em percentual**

A categoria que mais perde em dólares é o Mac (US$ 229,82), mas a que mais perde em percentual é o Watch (25,3%). O Mac perde mais dinheiro, mas em proporção perde menos: 14,3%.

Isso acontece porque as duas medidas respondem a perguntas diferentes. A queda em dólares mostra quanto valor se perdeu, e a queda em percentual mostra quanto se perdeu em relação ao preço de lançamento. O Mac lançou a US$ 1.612,31, então uma queda grande em dólares ainda é uma fatia pequena do preço. O Watch lançou a US$ 533,32, então uma queda menor em dólares (US$ 134,68) pesa bem mais em relação ao preço.

Uma categoria que perde muitos dólares não é necessariamente a que perde mais em percentual, porque o preço de lançamento muda o peso da queda. Por isso vale olhar as duas colunas, `queda_usd` e `queda_pct`, antes de tirar conclusões.

## Exercício 9: várias agregações com nome
Objetivo: uma tabela por categoria com preco_medio, preco_min, preco_max e anuncios, todos sobre Current_Price_USD.

💡 .agg(nome_novo=("coluna", "agregação"), ...).

✅ Um DataFrame de 4 linhas e 4 colunas.

In [140]:
df_resumo = df_apple.groupby("Product_Category").agg(preco_medio = ("Current_Price_USD", "mean"),
                                         preco_min = ("Current_Price_USD", "min"),
                                         preco_max = ("Current_Price_USD", "max"),
                                         anuncios = ("Current_Price_USD", "size")).round(2)

df_resumo

,preco_medio,preco_min,preco_max,anuncios
Product_Category,,,,
Mac,1382.49,332.76,2038.97,18020
Watch,398.64,137.76,814.94,17865
iPad,573.08,109.93,1222.89,15526
iPhone,758.67,226.61,1222.98,28589


## Exercício 10: conta entre dois resultados
Objetivo: que percentual dos anúncios de cada categoria está esgotado (Stock_Status == "Out of Stock")?

💡 Dois groupby(...).size(): um no DataFrame filtrado (só esgotados), outro na base inteira. Divida um pelo outro. É o terceiro caso da tabela, o mesmo raciocínio do exercício 14 do fundão.

✅ Uma Series de 4 percentuais. Qual categoria esgota mais?

In [149]:
filtro_stock_status = df_apple["Stock_Status"] == "Out of Stock"
df_apple_out_of_stock = df_apple[filtro_stock_status]
series_anuncios = df_apple.groupby("Product_Category").size()
series_out_of_stock = df_apple_out_of_stock.groupby("Product_Category").size()

pct_esgotado  = (series_out_of_stock/series_anuncios)*100
pct_esgotado .round(1)

Product_Category
Mac       15.9
Watch     16.0
iPad      19.8
iPhone    16.3
dtype: float64

O iPad é a categoria que mais esgota: 19,8% dos anúncios estão em "Out of Stock". As outras três ficam bem próximas entre si, entre 15,9% (Mac) e 16,3% (iPhone).